# NB5: K-Path Descriptor Extraction (206-row dataset)

**Goal:** Extract k-path-specific descriptors for every row in rashba.csv (206 entries),
combining them with existing compound-level DOS + elemental descriptors.
Output: ONE unified CSV with all descriptors, ready for feature selection in NB6.

**Location:** `Keshav-DDP/k-path/nb5_kpath_descriptors.ipynb`

## Descriptor Groups

| Group | Source | Varies per row? | ~Count |
|-------|--------|----------------|--------|
| CSV direct | rashba.csv (band, bandgap, ehull) | YES/NO | 3 |
| K-path geometry | POSCAR lattice + kpath labels | YES | ~11 |
| Degeneracy | space group + k-point symmetry | YES | ~4 |
| Structural env @ START | RDF + ADF at k-path start point | YES | ~14 |
| Structural env @ MID | RDF + ADF at k-path midpoint | YES | ~14 |
| Structural env @ END | RDF + ADF at k-path end point | YES | ~14 |
| Elemental properties | POSCAR elements | NO (per uid) | ~10 |
| DOS descriptors | existing nb1 output | NO (per uid) | ~3 |
| **Total** | | | **~73** |

NB6 will handle feature selection to find the best 5-8 from these ~73.

## Cell 1: Configuration -- ALL PATHS HERE

**IMPORTANT:** If you move this notebook, update these paths.
This notebook lives in: `Keshav-DDP/k-path/`
All input data is accessed relative to BASE_DIR (one level up).

In [ ]:
import os

# =============================================================================
# PATHS -- CHANGE THESE IF YOU MOVE THE NOTEBOOK
# =============================================================================

# This notebook lives in: Keshav-DDP/k-path/
# BASE_DIR should point to Keshav-DDP/
BASE_DIR = os.path.abspath("..")

# Input: rashba.csv with 206 rows
# Columns expected: Formula, uid, spacegroup, ehull, bandgap, band, kpath,
#                   Rashba_parameter, SS, dE, anticrossing
RASHBA_CSV = os.path.join(BASE_DIR, "Data", "rashba.csv")

# Input: compound folders with POSCAR and vasprun.xml
# Each folder is named like: AsBrTe-671e6de2497a
# POSCAR path inside: ss_2d%2F{folder}%2Fbands_ncl%2FPOSCAR
VASP_DIR = os.path.join(BASE_DIR, "Inverse-design", "rashba")

# Input: existing DOS descriptor CSV (99 rows, has E_pfrac_VBM, E_pfrac_CBM)
# From nb1 output. We merge on uid to get DOS features for each row.
# Set to None if not available (DOS features will be skipped).
EXISTING_DOS_CSV = os.path.join(
    BASE_DIR, "Weight-contribution", "contribution-model",
    "best_combo_0_w10_radius_mean.csv"
)

# Output: everything goes into this folder
RESULTS_DIR = os.path.join(".", "nb5_kpath_descriptors-results")
OUTPUT_CSV = os.path.join(RESULTS_DIR, "rashba_206_all_descriptors.csv")

# =============================================================================
# Print and verify paths
# =============================================================================
print("=" * 65)
print("  PATH CONFIGURATION")
print("=" * 65)
paths_to_check = [
    ("BASE_DIR", BASE_DIR),
    ("RASHBA_CSV", RASHBA_CSV),
    ("VASP_DIR", VASP_DIR),
    ("EXISTING_DOS_CSV", EXISTING_DOS_CSV),
    ("RESULTS_DIR", RESULTS_DIR),
    ("OUTPUT_CSV", OUTPUT_CSV),
]
for name, path in paths_to_check:
    exists = os.path.exists(path) if name != "OUTPUT_CSV" else "will be created"
    status = "OK" if exists == True else ("OUTPUT" if exists == "will be created" else "MISSING")
    print(f"  [{status:7s}] {name:20s} = {path}")

os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"\n  All outputs -> {RESULTS_DIR}/")

## Cell 2: Imports

In [ ]:
import pandas as pd
import numpy as np
import glob
import re
import warnings
warnings.filterwarnings('ignore')

from pymatgen.core.structure import Structure
from pymatgen.core.periodic_table import Element
from pymatgen.core.lattice import Lattice
from pymatgen.symmetry.bandstructure import HighSymmKpath
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer

print("Imports OK. pymatgen version:", end=" ")
try:
    import pymatgen
    print(pymatgen.__version__)
except:
    print("unknown")

## Cell 3: Load rashba.csv and inspect

In [ ]:
df = pd.read_csv(RASHBA_CSV)
print(f"Loaded rashba.csv: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Columns: {list(df.columns)}")
print(f"\nUnique UIDs: {df['uid'].nunique()}")
print(f"Unique Formulas: {df['Formula'].nunique()}")
print(f"\nUnique k-paths: {sorted(df['kpath'].unique())}")
print(f"\nBand distribution:\n{df['band'].value_counts().to_string()}")
print(f"\nSpace groups:\n{df['spacegroup'].value_counts().to_string()}")
print(f"\nEntries per compound (top 10):")
print(df.groupby('uid').size().sort_values(ascending=False).head(10).to_string())
print(f"\nRashba_parameter stats:\n{df['Rashba_parameter'].describe().to_string()}")

## Cell 4: Build compound lookup (uid -> folder, structure, k-point coords)

Uses pymatgen's `HighSymmKpath` to get correct high-symmetry k-point
coordinates for ANY space group (handles P3m1, P-1, Pm, Pmn2_1, etc.)

In [ ]:
def find_compound_folder(uid, vasp_dir):
    """Find the folder for a given uid in the VASP directory."""
    for folder in glob.glob(os.path.join(vasp_dir, "*")):
        folder_name = os.path.basename(folder)
        idx = folder_name.rfind('-')
        if idx != -1 and folder_name[idx+1:] == uid:
            return folder
    return None


def load_structure(compound_folder):
    """Load structure from POSCAR in compound folder."""
    # Try the encoded path first (ss_2d%2F...%2FPOSCAR)
    matches = glob.glob(os.path.join(compound_folder, "ss_2d*POSCAR"))
    if matches:
        return Structure.from_file(matches[0])
    # Try direct POSCAR
    direct = os.path.join(compound_folder, "POSCAR")
    if os.path.exists(direct):
        return Structure.from_file(direct)
    return None


def get_highsymm_kpoints(structure):
    """
    Get high-symmetry k-point coordinates using pymatgen HighSymmKpath.
    Works for ANY space group -- no manual lookup tables needed.
    Returns dict like {'G': array([0,0,0]), 'M': array([0.5,0,0]), ...}
    """
    try:
        kpath_obj = HighSymmKpath(structure)
        kpoints = kpath_obj.kpath["kpoints"]
        # kpoints is dict of label -> fractional coords
        return {label: np.array(coords) for label, coords in kpoints.items()}
    except Exception as e:
        print(f"  WARNING: HighSymmKpath failed: {e}. Using Gamma only.")
        return {'G': np.array([0.0, 0.0, 0.0])}


# Build lookup for all compounds
print("Building compound lookup (loading POSCARs + computing high-symmetry k-points)...")
compound_data = {}
missing_compounds = []

uid_list = df['uid'].unique()
for i, uid in enumerate(uid_list):
    folder = find_compound_folder(uid, VASP_DIR)
    if folder is None:
        missing_compounds.append(uid)
        continue

    structure = load_structure(folder)
    if structure is None:
        missing_compounds.append(uid)
        continue

    kpoints = get_highsymm_kpoints(structure)

    compound_data[uid] = {
        'folder': folder,
        'structure': structure,
        'lattice': structure.lattice,
        'reciprocal_lattice': structure.lattice.reciprocal_lattice,
        'kpoints': kpoints,  # high-symmetry k-point coords for this structure
    }

    if (i + 1) % 20 == 0:
        print(f"  Loaded {i+1}/{len(uid_list)} compounds...")

print(f"\nLoaded {len(compound_data)} / {len(uid_list)} compounds")
if missing_compounds:
    print(f"Missing ({len(missing_compounds)}): {missing_compounds[:10]}{'...' if len(missing_compounds) > 10 else ''}")

# Show k-point labels for a few compounds to verify
print("\nSample k-point mappings:")
for uid in list(compound_data.keys())[:3]:
    kpts = compound_data[uid]['kpoints']
    formula = df[df['uid'] == uid]['Formula'].iloc[0]
    sg = df[df['uid'] == uid]['spacegroup'].iloc[0]
    labels = list(kpts.keys())
    print(f"  {formula} ({sg}): {labels}")

## Cell 5: K-path parsing and coordinate resolution

In [ ]:
def parse_kpath(kpath_str):
    """Parse 'G->M' or 'M->K' into (start_label, end_label)."""
    for sep in ['->', '-', ' to ', '_']:
        if sep in kpath_str:
            parts = kpath_str.split(sep)
            if len(parts) == 2:
                return parts[0].strip(), parts[1].strip()
    raise ValueError(f"Cannot parse kpath: {kpath_str}")


# Common label aliases (rashba.csv might use different labels than pymatgen)
LABEL_ALIASES = {
    'G': ['G', 'GAMMA', '\\Gamma', 'Gamma', 'GM'],
    'M': ['M'],
    'K': ['K'],
    'X': ['X'],
    'Y': ['Y'],
    'S': ['S'],
    'A': ['A'],
    'L': ['L'],
    'H': ['H'],
    'Z': ['Z'],
    'R': ['R'],
    'T': ['T'],
    'U': ['U'],
    'N': ['N'],
    'P': ['P'],
}

# Build reverse lookup: alias -> canonical
ALIAS_TO_CANONICAL = {}
for canonical, aliases in LABEL_ALIASES.items():
    for alias in aliases:
        ALIAS_TO_CANONICAL[alias.upper()] = canonical


def resolve_kpoint_coords(label, compound_kpoints):
    """
    Resolve a k-point label to fractional coordinates.
    Uses the compound's HighSymmKpath result, with alias matching.
    """
    # Direct match
    if label in compound_kpoints:
        return compound_kpoints[label]

    # Try uppercase
    label_up = label.upper()
    for kp_label, coords in compound_kpoints.items():
        if kp_label.upper() == label_up:
            return coords

    # Try canonical alias
    canonical = ALIAS_TO_CANONICAL.get(label_up, label_up)
    if canonical in compound_kpoints:
        return compound_kpoints[canonical]
    for kp_label, coords in compound_kpoints.items():
        if kp_label.upper() == canonical:
            return coords

    # Special case: G/Gamma is always (0,0,0)
    if label_up in ['G', 'GAMMA', '\\GAMMA']:
        return np.array([0.0, 0.0, 0.0])

    # Last resort: warn and return origin
    print(f"    WARNING: Could not resolve k-point '{label}'. Available: {list(compound_kpoints.keys())}. Using (0,0,0).")
    return np.array([0.0, 0.0, 0.0])


# Test: resolve all k-paths in the dataset
print("Testing k-path resolution:")
unresolved = set()
for _, row in df.iterrows():
    uid = row['uid']
    if uid not in compound_data:
        continue
    start_l, end_l = parse_kpath(row['kpath'])
    kpts = compound_data[uid]['kpoints']
    for label in [start_l, end_l]:
        coords = resolve_kpoint_coords(label, kpts)
        if np.allclose(coords, 0) and label.upper() not in ['G', 'GAMMA']:
            unresolved.add(label)

if unresolved:
    print(f"  Unresolved labels: {unresolved}")
else:
    print("  All k-point labels resolved successfully!")

# Show resolution examples
for i in range(min(6, len(df))):
    row = df.iloc[i]
    uid = row['uid']
    if uid not in compound_data:
        continue
    start_l, end_l = parse_kpath(row['kpath'])
    kpts = compound_data[uid]['kpoints']
    sc = resolve_kpoint_coords(start_l, kpts)
    ec = resolve_kpoint_coords(end_l, kpts)
    print(f"  {row['Formula']} {row['kpath']}: {start_l}={sc} -> {end_l}={ec}")

## Cell 6: K-path geometry descriptors

In [ ]:
def extract_kpath_geometry(row, compound_data):
    """
    Extract k-path geometry descriptors for one row.
    Uses actual lattice vectors for real-space distances.
    """
    uid = row['uid']
    nan_result = {k: np.nan for k in [
        'kpath_real_distance', 'kpath_angle_deg',
        'kvec_x', 'kvec_y', 'kvec_z',
        'kstart_frac_x', 'kstart_frac_y', 'kstart_frac_z',
        'kend_frac_x', 'kend_frac_y', 'kend_frac_z',
        'kmid_frac_x', 'kmid_frac_y', 'kmid_frac_z',
    ]}

    if uid not in compound_data:
        return nan_result

    cdata = compound_data[uid]
    rec_lat = cdata['reciprocal_lattice']
    kpoints = cdata['kpoints']

    start_label, end_label = parse_kpath(row['kpath'])

    # Get fractional reciprocal coordinates
    k_start_frac = resolve_kpoint_coords(start_label, kpoints)
    k_end_frac = resolve_kpoint_coords(end_label, kpoints)
    k_mid_frac = (k_start_frac + k_end_frac) / 2.0

    # Convert to Cartesian reciprocal space (inverse Angstroms)
    k_start_cart = rec_lat.get_cartesian_coords(k_start_frac)
    k_end_cart = rec_lat.get_cartesian_coords(k_end_frac)

    # K-vector and distance
    kvec = k_end_cart - k_start_cart
    kpath_distance = np.linalg.norm(kvec)

    # Angle of k-vector in the xy-plane (2D-relevant)
    kpath_angle = np.degrees(np.arctan2(kvec[1], kvec[0])) if kpath_distance > 1e-10 else 0.0

    return {
        'kpath_real_distance': kpath_distance,
        'kpath_angle_deg': kpath_angle,
        'kvec_x': kvec[0], 'kvec_y': kvec[1], 'kvec_z': kvec[2],
        'kstart_frac_x': k_start_frac[0], 'kstart_frac_y': k_start_frac[1],
        'kstart_frac_z': k_start_frac[2],
        'kend_frac_x': k_end_frac[0], 'kend_frac_y': k_end_frac[1],
        'kend_frac_z': k_end_frac[2],
        'kmid_frac_x': k_mid_frac[0], 'kmid_frac_y': k_mid_frac[1],
        'kmid_frac_z': k_mid_frac[2],
    }


# Test
for i in range(min(4, len(df))):
    row = df.iloc[i]
    result = extract_kpath_geometry(row, compound_data)
    print(f"{row['Formula']} {row['kpath']}: dist={result['kpath_real_distance']:.4f} A^-1, "
          f"angle={result['kpath_angle_deg']:.1f} deg")

## Cell 7: Degeneracy descriptors

In [ ]:
def get_kpoint_degeneracy(label, structure, kpoints):
    """
    Get degeneracy (multiplicity) of a k-point using pymatgen's symmetry analysis.
    Degeneracy = how many equivalent k-points exist in the full Brillouin zone.
    """
    coords = resolve_kpoint_coords(label, kpoints)

    # Gamma is always degeneracy 1
    if np.allclose(coords, 0):
        return 1

    # Use SpacegroupAnalyzer to get symmetry operations
    try:
        sga = SpacegroupAnalyzer(structure)
        symmops = sga.get_symmetry_operations()

        # Apply all symmetry ops to the k-point, count unique images
        unique_kpoints = set()
        for op in symmops:
            rotated = op.apply_rotation_only(coords)
            # Round to avoid floating point duplicates
            rounded = tuple(np.round(rotated % 1.0, decimals=6))
            unique_kpoints.add(rounded)

        return len(unique_kpoints)
    except Exception:
        # Fallback: estimate from label
        fallback = {'G': 1, 'M': 3, 'K': 2, 'X': 2, 'Y': 2, 'S': 4}
        return fallback.get(label.upper(), 1)


def extract_degeneracy(row, compound_data):
    """Extract degeneracy features for a row."""
    uid = row['uid']
    if uid not in compound_data:
        return {k: np.nan for k in [
            'kstart_degeneracy', 'kend_degeneracy',
            'kpath_interior_degeneracy', 'kpath_total_weight'
        ]}

    cdata = compound_data[uid]
    start_label, end_label = parse_kpath(row['kpath'])

    d_start = get_kpoint_degeneracy(start_label, cdata['structure'], cdata['kpoints'])
    d_end = get_kpoint_degeneracy(end_label, cdata['structure'], cdata['kpoints'])

    # Interior: points between high-symmetry points have double the max endpoint
    # degeneracy (from prof's explanation: inside BZ, not shared with neighbors)
    d_interior = 2 * max(d_start, d_end)

    return {
        'kstart_degeneracy': d_start,
        'kend_degeneracy': d_end,
        'kpath_interior_degeneracy': d_interior,
        'kpath_total_weight': d_start + d_end + d_interior,
    }


# Test
for i in range(min(4, len(df))):
    row = df.iloc[i]
    result = extract_degeneracy(row, compound_data)
    print(f"{row['Formula']} {row['kpath']}: {result}")

## Cell 8: Structural environment descriptors (RDF + ADF)

Computed at THREE points along the k-path: start, midpoint, and end.
For each point we compute:
- RDF-based: coordination, density sums (plain, Z-weighted, Z^4-weighted, mass-weighted)
- ADF-based: angular distribution statistics around the point
- Nearest atom features: distance, Z, mass

In [ ]:
def kfrac_to_realspace(k_frac, structure):
    """
    Map fractional reciprocal coordinate to a real-space position in the unit cell.

    Heuristic: treat the fractional k-coords as fractional positions in the unit cell.
    This gives a real-space point that roughly corresponds to the spatial region
    the k-vector probes. Not exact (k-space and real-space are Fourier duals)
    but useful for generating environment descriptors.
    """
    frac = np.mod(k_frac, 1.0)
    return structure.lattice.get_cartesian_coords(frac)


def compute_rdf_descriptors(point_cart, structure, cutoffs=[2.0, 4.0]):
    """
    Compute radial distribution function descriptors around a Cartesian point.

    For each cutoff:
    - coordination: number of atoms within cutoff
    - rdf_sum: sum(1/r) for atoms within cutoff
    - rdf_Z_weighted: sum(Z/r)
    - rdf_Z4_weighted: sum(Z^4/r)
    - rdf_mass_weighted: sum(mass/r)
    - mean_Z, max_Z within cutoff

    Also: nearest atom distance, Z, mass, Z^4.
    """
    features = {}
    frac_point = structure.lattice.get_fractional_coords(point_cart)

    # Compute distances to all sites (with periodic images)
    atom_data = []
    for site in structure:
        d = structure.lattice.get_all_distances([frac_point], [site.frac_coords])[0][0]
        z = site.specie.Z
        mass = float(site.specie.atomic_mass)
        atom_data.append((d, z, mass))

    atom_data.sort(key=lambda x: x[0])

    # Nearest atom
    if atom_data:
        nd, nz, nm = atom_data[0]
        features['nearest_dist'] = nd
        features['nearest_Z'] = nz
        features['nearest_mass'] = nm
        features['nearest_Z4'] = nz ** 4
    else:
        features['nearest_dist'] = np.nan
        features['nearest_Z'] = np.nan
        features['nearest_mass'] = np.nan
        features['nearest_Z4'] = np.nan

    # Cutoff-based features
    for cutoff in cutoffs:
        s = f"_{cutoff:.0f}A"
        atoms = [(d, z, m) for d, z, m in atom_data if 0 < d <= cutoff]
        n = len(atoms)

        features[f'coord{s}'] = n
        if n > 0:
            features[f'rdf_sum{s}'] = sum(1.0/d for d, z, m in atoms)
            features[f'rdf_Z{s}'] = sum(z/d for d, z, m in atoms)
            features[f'rdf_Z4{s}'] = sum((z**4)/d for d, z, m in atoms)
            features[f'rdf_mass{s}'] = sum(m/d for d, z, m in atoms)
            features[f'mean_Z{s}'] = np.mean([z for d, z, m in atoms])
            features[f'max_Z{s}'] = max(z for d, z, m in atoms)
        else:
            for p in ['rdf_sum', 'rdf_Z', 'rdf_Z4', 'rdf_mass', 'mean_Z', 'max_Z']:
                features[f'{p}{s}'] = 0.0

    return features


def compute_adf_descriptors(point_cart, structure, cutoff=4.0):
    """
    Compute angular distribution function descriptors around a point.

    For all atom pairs (i, j) within cutoff of the point, compute
    the angle point-i-j and point-j-i. This captures the angular
    arrangement of atoms around the k-path point.

    Returns: mean, std, min, max of angles; number of triplets.
    """
    features = {}
    frac_point = structure.lattice.get_fractional_coords(point_cart)

    # Get neighbor atoms within cutoff
    neighbors = []
    for site in structure:
        d = structure.lattice.get_all_distances([frac_point], [site.frac_coords])[0][0]
        if 0 < d <= cutoff:
            cart = site.coords
            neighbors.append((cart, d))

    if len(neighbors) < 2:
        features['adf_mean'] = np.nan
        features['adf_std'] = np.nan
        features['adf_min'] = np.nan
        features['adf_max'] = np.nan
        features['adf_n_triplets'] = 0
        return features

    # Compute angles between all pairs as seen from the central point
    angles = []
    for i in range(len(neighbors)):
        for j in range(i+1, len(neighbors)):
            vec_i = neighbors[i][0] - point_cart
            vec_j = neighbors[j][0] - point_cart
            norm_i = np.linalg.norm(vec_i)
            norm_j = np.linalg.norm(vec_j)
            if norm_i > 1e-10 and norm_j > 1e-10:
                cos_angle = np.clip(np.dot(vec_i, vec_j) / (norm_i * norm_j), -1, 1)
                angle_deg = np.degrees(np.arccos(cos_angle))
                angles.append(angle_deg)

    if angles:
        features['adf_mean'] = np.mean(angles)
        features['adf_std'] = np.std(angles)
        features['adf_min'] = np.min(angles)
        features['adf_max'] = np.max(angles)
        features['adf_n_triplets'] = len(angles)
    else:
        features['adf_mean'] = np.nan
        features['adf_std'] = np.nan
        features['adf_min'] = np.nan
        features['adf_max'] = np.nan
        features['adf_n_triplets'] = 0

    return features


def compute_point_environment(point_frac, structure, prefix, cutoffs=[2.0, 4.0]):
    """
    Full environment descriptor for one point.
    Combines RDF + ADF with a prefix like 'start_', 'mid_', 'end_'.
    """
    point_cart = kfrac_to_realspace(point_frac, structure)
    rdf = compute_rdf_descriptors(point_cart, structure, cutoffs=cutoffs)
    adf = compute_adf_descriptors(point_cart, structure, cutoff=max(cutoffs))

    # Add prefix to all keys
    features = {}
    for k, v in rdf.items():
        features[f'{prefix}{k}'] = v
    for k, v in adf.items():
        features[f'{prefix}{k}'] = v

    return features


# Test on one compound
test_uid = df.iloc[0]['uid']
if test_uid in compound_data:
    cdata = compound_data[test_uid]
    start_l, end_l = parse_kpath(df.iloc[0]['kpath'])
    k_start = resolve_kpoint_coords(start_l, cdata['kpoints'])
    k_end = resolve_kpoint_coords(end_l, cdata['kpoints'])
    k_mid = (k_start + k_end) / 2.0

    print(f"Test: {df.iloc[0]['Formula']} {df.iloc[0]['kpath']}")
    for prefix, kfrac in [('start_', k_start), ('mid_', k_mid), ('end_', k_end)]:
        env = compute_point_environment(kfrac, cdata['structure'], prefix)
        print(f"\n  {prefix.upper()} ({len(env)} features):")
        for k, v in list(env.items())[:5]:
            print(f"    {k}: {v:.4f}" if isinstance(v, (float, np.floating)) else f"    {k}: {v}")
        print(f"    ... and {len(env) - 5} more")

## Cell 9: Elemental property descriptors (from POSCAR)

In [ ]:
def compute_elemental_properties(structure):
    """Compute elemental property descriptors from pymatgen Structure."""
    elements = list(set(structure.species))

    atomic_nums = [el.Z for el in elements]
    masses = [float(el.atomic_mass) for el in elements]
    radii = [float(el.atomic_radius) if el.atomic_radius else 1.0 for el in elements]
    electroneg = [float(el.X) if el.X else 2.0 for el in elements]

    return {
        'max_Z': max(atomic_nums),
        'max_Z4': max(z**4 for z in atomic_nums),
        'max_mass': max(masses),
        'min_mass': min(masses),
        'radius_mean': np.mean(radii),
        'radius_diff': max(radii) - min(radii) if len(radii) > 1 else 0.0,
        'X_mean': np.mean(electroneg),
        'X_diff': max(electroneg) - min(electroneg) if len(electroneg) > 1 else 0.0,
        'n_elements': len(elements),
        'mean_Z': np.mean(atomic_nums),
        'sum_Z4': sum(z**4 for z in atomic_nums),
    }


# Precompute for all compounds
elemental_props = {}
for uid, cdata in compound_data.items():
    elemental_props[uid] = compute_elemental_properties(cdata['structure'])

print(f"Computed elemental properties for {len(elemental_props)} compounds")
example_uid = list(elemental_props.keys())[0]
formula = df[df['uid'] == example_uid]['Formula'].iloc[0]
print(f"\nExample: {formula} ({example_uid})")
for k, v in elemental_props[example_uid].items():
    print(f"  {k}: {v}")

## Cell 10: Load existing DOS descriptors

In [ ]:
dos_features_df = None

if EXISTING_DOS_CSV and os.path.exists(EXISTING_DOS_CSV):
    df_dos_raw = pd.read_csv(EXISTING_DOS_CSV)
    print(f"Loaded DOS CSV: {df_dos_raw.shape}")
    print(f"  Columns: {list(df_dos_raw.columns)}")

    # Find DOS columns and uid column
    dos_cols = [c for c in df_dos_raw.columns if 'pfrac' in c.lower() or 'E_' in c]
    uid_col = 'uid' if 'uid' in df_dos_raw.columns else None

    if uid_col and dos_cols:
        dos_features_df = df_dos_raw[[uid_col] + dos_cols].drop_duplicates(subset=[uid_col])
        print(f"  Will merge DOS features: {dos_cols}")
        print(f"  Unique UIDs in DOS CSV: {dos_features_df[uid_col].nunique()}")
    else:
        print(f"  WARNING: Could not find uid ({uid_col}) or DOS columns ({dos_cols})")
        print(f"  You may need to adjust column detection above.")
else:
    print(f"No DOS CSV found at: {EXISTING_DOS_CSV}")
    print("  DOS features will NOT be included. To add them later:")
    print("  1. Set EXISTING_DOS_CSV to the correct path in Cell 1")
    print("  2. Re-run this cell and Cell 12")

## Cell 11: MAIN EXTRACTION LOOP

This is the big cell. For each of the 206 rows, extract ALL descriptors.
Takes a few minutes because of structure environment calculations.

In [ ]:
print("=" * 65)
print("  EXTRACTING ALL DESCRIPTORS FOR 206 ROWS")
print("=" * 65)

all_rows = []
errors = []

for idx, row in df.iterrows():
    uid = row['uid']
    features = {}

    # === Identifiers (kept for tracking, not used as features) ===
    features['Formula'] = row['Formula']
    features['uid'] = uid
    features['kpath'] = row['kpath']
    features['Rashba_parameter'] = row['Rashba_parameter']

    # === Group 1: Direct from CSV ===
    features['band_binary'] = 1 if row['band'] == 'C' else 0
    features['bandgap'] = row.get('bandgap', np.nan)
    features['ehull'] = row.get('ehull', np.nan)

    # === Group 2: K-path geometry ===
    geom = extract_kpath_geometry(row, compound_data)
    features.update(geom)

    # === Group 3: Degeneracy ===
    degen = extract_degeneracy(row, compound_data)
    features.update(degen)

    # === Group 4: Structural environment at START, MID, END ===
    if uid in compound_data:
        cdata = compound_data[uid]
        start_label, end_label = parse_kpath(row['kpath'])
        k_start = resolve_kpoint_coords(start_label, cdata['kpoints'])
        k_end = resolve_kpoint_coords(end_label, cdata['kpoints'])
        k_mid = (k_start + k_end) / 2.0

        try:
            features.update(compute_point_environment(k_start, cdata['structure'], 'start_'))
            features.update(compute_point_environment(k_mid, cdata['structure'], 'mid_'))
            features.update(compute_point_environment(k_end, cdata['structure'], 'end_'))
        except Exception as e:
            errors.append((idx, uid, f"env descriptors: {e}"))
    else:
        errors.append((idx, uid, "compound not found"))

    # === Group 5: Elemental properties ===
    if uid in elemental_props:
        features.update(elemental_props[uid])

    all_rows.append(features)

    if (idx + 1) % 25 == 0:
        print(f"  Processed {idx + 1}/{len(df)} rows...")

df_full = pd.DataFrame(all_rows)
print(f"\nExtraction complete: {df_full.shape[0]} rows, {df_full.shape[1]} columns")

if errors:
    print(f"\nErrors ({len(errors)}):")
    for e in errors[:10]:
        print(f"  Row {e[0]}, uid={e[1]}: {e[2]}")
    if len(errors) > 10:
        print(f"  ... and {len(errors) - 10} more")

## Cell 12: Merge DOS features and create derived features

In [ ]:
# Merge DOS descriptors from 99-row CSV
if dos_features_df is not None:
    before_cols = set(df_full.columns)
    df_full = df_full.merge(dos_features_df, on='uid', how='left')
    new_cols = set(df_full.columns) - before_cols
    print(f"Merged DOS features: {sorted(new_cols)}")
else:
    print("Skipping DOS merge (no DOS CSV loaded).")

# Create E_pfrac_relevant: picks VBM or CBM based on band type
if 'E_pfrac_VBM' in df_full.columns and 'E_pfrac_CBM' in df_full.columns:
    df_full['E_pfrac_relevant'] = np.where(
        df_full['band_binary'] == 0,
        df_full['E_pfrac_VBM'],
        df_full['E_pfrac_CBM']
    )
    print("Created E_pfrac_relevant")

print(f"\nFinal shape: {df_full.shape[0]} rows, {df_full.shape[1]} columns")

## Cell 13: Save the unified CSV

In [ ]:
# Identify columns
id_cols = ['Formula', 'uid', 'kpath', 'Rashba_parameter']
feature_cols = sorted([c for c in df_full.columns if c not in id_cols])

print(f"Identifier columns ({len(id_cols)}): {id_cols}")
print(f"\nFeature columns ({len(feature_cols)}):")
for i, col in enumerate(feature_cols):
    n_unique = df_full[col].nunique()
    n_nan = df_full[col].isna().sum()
    dtype = df_full[col].dtype
    print(f"  {i+1:3d}. {col:45s} unique={n_unique:4d}, NaN={n_nan:3d}, dtype={dtype}")

# Save full CSV
df_full.to_csv(OUTPUT_CSV, index=False)
print(f"\nSaved: {OUTPUT_CSV}")
print(f"  {df_full.shape[0]} rows, {df_full.shape[1]} columns")

## Cell 14: Quick sanity checks

In [ ]:
print("=" * 65)
print("  SANITY CHECKS")
print("=" * 65)

# 1. Check which features vary within same compound
print("\n1. Features that VARY within same compound (k-path specific):")
multi_uids = df_full.groupby('uid').filter(lambda g: len(g) > 1)['uid'].unique()
varying_features = []
constant_features = []

for col in feature_cols:
    if df_full[col].dtype in ['float64', 'int64', 'float32', 'int32']:
        n_varying = 0
        for uid in multi_uids[:30]:
            subset = df_full[df_full['uid'] == uid][col]
            if subset.nunique() > 1:
                n_varying += 1
        pct = n_varying / min(30, len(multi_uids)) * 100
        if pct > 10:
            varying_features.append((col, pct))
        else:
            constant_features.append(col)

print(f"  K-path specific ({len(varying_features)} features):")
for col, pct in sorted(varying_features, key=lambda x: -x[1]):
    print(f"    {col:45s}: varies in {pct:.0f}% of multi-entry compounds")

print(f"\n  Compound-level only ({len(constant_features)} features):")
for col in constant_features[:10]:
    print(f"    {col}")
if len(constant_features) > 10:
    print(f"    ... and {len(constant_features) - 10} more")

# 2. Constant columns
print("\n2. Constant columns (remove before modeling):")
for col in feature_cols:
    if df_full[col].nunique() <= 1:
        print(f"  REMOVE: {col} (only {df_full[col].nunique()} unique value)")

# 3. High NaN columns
print("\n3. Columns with >10% NaN:")
for col in feature_cols:
    pct_nan = df_full[col].isna().mean() * 100
    if pct_nan > 10:
        print(f"  {col}: {pct_nan:.1f}% NaN")

# 4. Quick single-feature XGBoost R2 (rough sanity check)
print("\n4. Single-feature XGBoost LOO R2 (top 15):")
print("   (Quick check only -- proper feature selection in NB6)")
try:
    from xgboost import XGBRegressor
    from sklearn.model_selection import LeaveOneOut, cross_val_score

    target = df_full['Rashba_parameter'].values
    single_r2 = {}

    for col in feature_cols:
        if df_full[col].dtype not in ['float64', 'int64', 'float32', 'int32']:
            continue
        if df_full[col].isna().sum() > 10:
            continue
        if df_full[col].nunique() <= 1:
            continue

        X_single = df_full[[col]].fillna(0).values
        model = XGBRegressor(n_estimators=50, max_depth=3, learning_rate=0.1,
                             random_state=42, verbosity=0)
        scores = cross_val_score(model, X_single, target, cv=5, scoring='r2')
        single_r2[col] = np.mean(scores)

    for col, r2 in sorted(single_r2.items(), key=lambda x: -x[1])[:15]:
        marker = " <-- k-path specific" if col in [c for c, _ in varying_features] else ""
        print(f"    {col:45s}: R2 = {r2:.3f}{marker}")

except ImportError:
    print("  XGBoost not installed, skipping single-feature check.")
    print("  pip install xgboost")

## Cell 15: Summary

In [ ]:
print("=" * 65)
print("  NB5 COMPLETE -- SUMMARY")
print("=" * 65)
print(f"""
  Dataset:    {df_full.shape[0]} rows (was 99 with max aggregation)
  Features:   {len(feature_cols)} total
    - K-path specific: {len(varying_features)} (vary within compound)
    - Compound-level:  {len(constant_features)} (same for all rows of a uid)

  Output CSV: {OUTPUT_CSV}

  Descriptor groups:
    - CSV direct:      band_binary, bandgap, ehull
    - K-path geometry:  kpath_real_distance, angle, kvec, frac coords
    - Degeneracy:       start/end/interior degeneracy, total weight
    - Struct env START: RDF + ADF at k-path start point
    - Struct env MID:   RDF + ADF at k-path midpoint
    - Struct env END:   RDF + ADF at k-path end point
    - Elemental:        radius, Z, Z4, mass, electronegativity
    - DOS:              E_pfrac_VBM, E_pfrac_CBM, E_pfrac_relevant

  Next step: NB6 -- Feature selection + modeling
    - Sequential forward selection (sklearn SequentialFeatureSelector)
    - SHAP analysis on full model
    - Exhaustive 1-feature, 2-feature, 3-feature combos with XGBoost
    - Find optimal 5-8 feature subset
    - Run regression + classification on selected features
""")